In [1]:
!pip install xlrd
!pip install openpyxl


[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [18]:
import os
import pandas as pd
import re  # Import regex library

folder_path = "/workspaces/verbose-space-bassoon/Jakarta Utara/Kelapa Gading/12"  # Ganti dengan lokasi folder yang sesuai
excel_files = [f for f in os.listdir(folder_path) if f.endswith(('.xlsx', '.xls', '.xlsb'))]

daily_averages = []

for file in excel_files:
    file_path = os.path.join(folder_path, file)

    try:
        print(f"\nMemproses file: {file}")

        # Deteksi engine berdasarkan format file
        if file.endswith(".xlsx"):
            engine = "openpyxl"
        elif file.endswith(".xls"):
            engine = "xlrd"
        elif file.endswith(".xlsb"):
            engine = "pyxlsb"
        else:
            raise ValueError("Format file tidak didukung")

        # Baca file Excel
        df = pd.read_excel(file_path, engine=engine)

        # Cek kolom yang tersedia
        print(f"Kolom tersedia di {file}: {df.columns.tolist()}")

        expected_cols = ["ISPU PM10", "ISPU PM2.5", "ISPU SO2", "ISPU CO", "ISPU O3", "ISPU NO2"]
        if not all(col in df.columns for col in expected_cols):
            print(f"File {file} tidak memiliki semua kolom yang diharapkan, dilewati")
            continue

        # Cek isi awal dataframe sebelum konversi
        print(f"Data awal dalam {file}:\n", df.head())

        # Konversi ke numerik
        df[expected_cols] = df[expected_cols].apply(pd.to_numeric, errors='coerce')

        # Cek jumlah NaN setelah konversi
        print(f"Jumlah NaN setelah konversi di {file}:\n", df[expected_cols].isna().sum())

        # Hitung rata-rata
        avg_values = df[expected_cols].mean(skipna=True)

        # Cek apakah semua nilai adalah NaN
        if avg_values.isna().all():
            print(f"Semua nilai ISPU di file {file} adalah NaN, dilewati")
            continue

        # Ekstrak tanggal dari nama file dengan regex: "ISPU-DKI2-KELAPA-GADING-YYYY-MM-DD-..."
        date_match = re.search(r'(\d{4}-\d{2}-\d{2})', file)  # Mencocokkan pola YYYY-MM-DD

        if date_match:
            date_str = date_match.group(0)  # Ambil hasil yang cocok
            print(f"String tanggal yang diekstrak: {date_str}")  # Debugging: tampilkan tanggal yang diekstrak

            try:
                # Cek apakah format tanggal benar dengan 'pd.to_datetime'
                date = pd.to_datetime(date_str, format="%Y-%m-%d", errors="raise")  # Gunakan "raise" untuk error yang lebih eksplisit
                print(f"Tanggal yang diparsing: {date}")  # Debugging: tampilkan tanggal setelah parsing
            except Exception as e:
                print(f"Error parsing tanggal dari file {file}: {e}")
                date = pd.NaT
        else:
            print(f"Tanggal tidak ditemukan dalam nama file: {file}, dilewati")
            date = pd.NaT

        # Simpan hasil
        daily_averages.append([date] + avg_values.tolist())

    except Exception as e:
        print(f"Error membaca file {file}: {e}")

# Buat DataFrame hasil
df_daily = pd.DataFrame(daily_averages, columns=["Tanggal", "PM 10", "PM 2.5", "O2", "CO", "O3", "NO2"])
df_daily = df_daily.sort_values(by="Tanggal").reset_index(drop=True)

# Tampilkan hasil akhir
print("\nHasil Daily Averages:")
print(df_daily)

# Simpan DataFrame ke dalam file CSV
output_csv_path = "/workspaces/verbose-space-bassoon/Jakarta Utara/Kelapa Gading/12/daily_averages.csv"  # Ganti dengan path tempat kamu ingin menyimpan file CSV
df_daily.to_csv(output_csv_path, index=False)

print(f"\nData telah disimpan ke {output_csv_path}")



Memproses file: ISPU-DKI2-KELAPA-GADING-2023-12-21-20250128093317.xlsx
Kolom tersedia di ISPU-DKI2-KELAPA-GADING-2023-12-21-20250128093317.xlsx: ['Waktu', 'ISPU PM10', 'ISPU PM2.5', 'ISPU SO2', 'ISPU CO', 'ISPU O3', 'ISPU NO2', 'Status By PM10', 'Status By PM2.5', 'Status By SO2', 'Status By CO', 'Status By O3', 'Status By NO2', 'Critical Parameter', 'Overall ISPU Status']
Data awal dalam ISPU-DKI2-KELAPA-GADING-2023-12-21-20250128093317.xlsx:
    Waktu ISPU PM10 ISPU PM2.5 ISPU SO2 ISPU CO ISPU O3 ISPU NO2  \
0  00:00         -          -        -       -       -        -   
1  01:00         -          -        -       -       -        -   
2  02:00         -          -        -       -       -        -   
3  03:00        52         85       34       7      24       34   
4  04:00        52         86       34       8      24       34   

  Status By PM10 Status By PM2.5 Status By SO2 Status By CO Status By O3  \
0              -               -             -            -            

In [19]:
df_daily

,Tanggal,PM 10,PM 2.5,O2,CO,O3,NO2
0,2023-12-01,50.785714,66.000000,33.071429,9.857143,24.285714,34.285714
1,2023-12-02,62.652174,83.913043,34.391304,14.695652,22.347826,37.000000
2,2023-12-03,52.055556,67.333333,32.722222,8.222222,31.555556,24.222222
3,2023-12-04,42.400000,59.500000,32.550000,7.950000,22.650000,29.000000
4,2023-12-05,58.736842,77.526316,34.526316,15.736842,19.631579,32.000000
5,2023-12-06,58.761905,78.476190,33.666667,9.190476,31.857143,28.952381
6,2023-12-07,56.300000,74.900000,33.750000,10.000000,27.250000,34.450000
7,2023-12-08,55.842105,76.473684,32.526316,8.105263,31.157895,34.421053
8,2023-12-09,62.523810,84.666667,33.000000,13.000000,24.047619,39.571429
9,2023-12-10,54.850000,72.600000,32.950000,9.000000,27.300000,25.700000


In [1]:
import pandas as pd

# Tentukan path file Excel yang ingin dibaca
file_path = '/workspaces/verbose-space-bassoon/Jakarta Utara/Kelapa Gading/12/ISPU-DKI2-KELAPA-GADING-2023-12-03-20250128093239.xlsx'  # Ganti dengan path file yang sesuai

# Baca file Excel
df = pd.read_excel(file_path)

# Tampilkan beberapa baris pertama dari DataFrame
df.head()


,Waktu,ISPU PM10,ISPU PM2.5,ISPU SO2,ISPU CO,ISPU O3,ISPU NO2,Status By PM10,Status By PM2.5,Status By SO2,Status By CO,Status By O3,Status By NO2,Critical Parameter,Overall ISPU Status
0,00:00,68,89,35,12,34,30,Sedang,Sedang,Baik,Baik,Baik,Baik,pm25,Sedang
1,01:00,67,88,34,12,35,30,Sedang,Sedang,Baik,Baik,Baik,Baik,pm25,Sedang
2,02:00,67,85,34,11,34,29,Sedang,Sedang,Baik,Baik,Baik,Baik,pm25,Sedang
3,03:00,-,-,-,-,-,-,-,-,-,-,-,-,-,-
4,04:00,-,-,-,-,-,-,-,-,-,-,-,-,-,-


In [7]:
import pandas as pd

# Tentukan path file Excel yang ingin dibaca
file_path = '/workspaces/verbose-space-bassoon/Jakarta Utara/Kelapa Gading/Merged_Data.xlsx'  # Ganti dengan path file yang sesuai

# Baca file Excel
df = pd.read_excel(file_path)

# Tampilkan beberapa baris pertama dari DataFrame
print(df.head())


        Date      PM 10     PM 2.5         O2         CO         O3  \
0 2023-01-01  45.823529  60.764706  56.235294  15.117647  20.235294   
1 2023-01-02  35.789474  45.157895  58.315789  15.052632  22.315789   
2 2023-01-03  28.833333  33.500000  57.125000  12.958333  21.708333   
3 2023-01-04  29.947368  42.421053  59.157895  16.894737  19.947368   
4 2023-01-05  36.454545  53.909091  56.454545  25.272727  22.181818   

         NO2  TAVG RH_AVG     RR FF_AVG       Location  
0  17.470588  25.9     93     38      2  Jakarta_Utara  
1  19.263158  27.2     88  134.4      3  Jakarta_Utara  
2  16.375000  26.7     83    0.7      2  Jakarta_Utara  
3  25.000000    27     88    3.2      2  Jakarta_Utara  
4  34.636364  27.5     84   31.3      3  Jakarta_Utara  


In [8]:
df

,Date,PM 10,PM 2.5,O2,CO,O3,NO2,TAVG,RH_AVG,RR,FF_AVG,Location
0,2023-01-01,45.823529,60.764706,56.235294,15.117647,20.235294,17.470588,25.9,93,38,2,Jakarta_Utara
1,2023-01-02,35.789474,45.157895,58.315789,15.052632,22.315789,19.263158,27.2,88,134.4,3,Jakarta_Utara
2,2023-01-03,28.833333,33.500000,57.125000,12.958333,21.708333,16.375000,26.7,83,0.7,2,Jakarta_Utara
3,2023-01-04,29.947368,42.421053,59.157895,16.894737,19.947368,25.000000,27,88,3.2,2,Jakarta_Utara
4,2023-01-05,36.454545,53.909091,56.454545,25.272727,22.181818,34.636364,27.5,84,31.3,3,Jakarta_Utara
...,...,...,...,...,...,...,...,...,...,...,...,...
360,2023-12-27,58.714286,83.785714,36.642857,8.428571,27.142857,32.785714,29.4,80,0,1,Jakarta_Utara
361,2023-12-28,60.388889,79.555556,32.777778,11.222222,31.444444,35.500000,30.4,78,0,2,Jakarta_Utara
362,2023-12-29,63.421053,86.631579,34.000000,12.789474,32.263158,6.789474,29.8,79,0,1,Jakarta_Utara
363,2023-12-30,57.400000,83.450000,32.850000,18.550000,30.500000,18.950000,29.7,79,8888,1,Jakarta_Utara


In [6]:
import pandas as pd

# Path file
file_path = "/workspaces/verbose-space-bassoon/Jakarta Utara/Kelapa Gading/Merged_Data.xlsx"

# Baca file Excel
df = pd.read_excel(file_path)

# Tambahkan kolom 'Location' dengan nilai 'Jakarta Utara'
df["Location"] = "Jakarta_Utara"

# Hapus kolom 'Month' jika ada
if "Month" in df.columns:
    df.drop(columns=["Month"], inplace=True)

# Simpan kembali ke file yang sama
df.to_excel(file_path, index=False)

print(f"File berhasil diperbarui dan disimpan kembali di: {file_path}")


File berhasil diperbarui dan disimpan kembali di: /workspaces/verbose-space-bassoon/Jakarta Utara/Kelapa Gading/Merged_Data.xlsx
